In [1]:
from ori_model import *
from models import *
from lm_models import *

from ult import *



# tok

In [ ]:
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.normalizers import NFKC

# load dataset
dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")

# init tokenizer
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.normalizer = NFKC()
tokenizer.pre_tokenizer = Whitespace()

# trainer
trainer = BpeTrainer(
    vocab_size=3000,
    min_frequency=2,
    special_tokens=["[PAD]", "[UNK]", "[BOS]", "[EOS]"]
)

# train
tokenizer.train_from_iterator(
    (x["text"] for x in dataset if len(x["text"]) > 0),
    trainer=trainer
)

# save
tokenizer.save("bpe_3000.json")


# dataprocess

In [2]:
# =========================
# Step 1: load trained BPE tokenizer
# =========================
from transformers import PreTrainedTokenizerFast

tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="bpe_3000.json",
    bos_token="[BOS]",
    eos_token="[EOS]",
    unk_token="[UNK]",
    pad_token="[PAD]",
)

vocab_size = tokenizer.vocab_size
print(f"Loaded tokenizer, vocab size = {vocab_size}")


# =========================
# Step 2: load WikiText-2
# =========================
from datasets import load_dataset

raw = load_dataset("wikitext", "wikitext-2-raw-v1")

train_texts = raw["train"]["text"]
val_texts   = raw["validation"]["text"]

print(f"Train texts: {len(train_texts)}")
print(f"Val texts:   {len(val_texts)}")


# =========================
# Step 3: build LM Dataset
# =========================
import torch
from torch.utils.data import Dataset


class LMDataset(Dataset):
    def __init__(self, texts, tokenizer, block_size):
        ids = []

        for text in texts:
            if not text.strip():
                continue
            ids.extend(tokenizer.encode(text))

        # truncate to multiple of block_size
        n = (len(ids) // block_size) * block_size
        ids = ids[:n]

        self.data = torch.tensor(ids, dtype=torch.long).view(-1, block_size)

    def __len__(self):
        return self.data.size(0)

    def __getitem__(self, idx):
        return self.data[idx]


# -------------------------
# instantiate datasets
# -------------------------
block_size = 128

train_ds = LMDataset(train_texts, tokenizer, block_size)
val_ds   = LMDataset(val_texts, tokenizer, block_size)

print(f"Train LM blocks: {len(train_ds)}")
print(f"Val LM blocks:   {len(val_ds)}")

# sanity check
x = train_ds[0]
print("Sample shape:", x.shape)
print("Sample tokens:", x[:10])

c:\Users\jitie\pp1\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded tokenizer, vocab size = 3000
Train texts: 36718
Val texts:   3760
Train LM blocks: 25979
Val LM blocks:   2715
Sample shape: torch.Size([128])
Sample tokens: tensor([  32, 2863, 2810, 1067,   68, 1166, 1427, 1020, 1374, 1580])


# model

In [3]:
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")


## no-rope

### with base

In [4]:
models={
    "lm_po":GFAPMiniLM_PO(
    vocab_size=3000,
    d_model=16,
    num_layers=3,
    context_length=128,
  
    use_rope=False
).to(device),
  "lm_pf": GFAPMiniLM_PF(
    vocab_size=3000,
    d_model=16,
    num_layers=3,
    context_length=128,
  
    use_rope=False
).to(device)
,
    "lm_pok":GFAPMiniLM_POK(
    vocab_size=3000,
    d_model=16,
    num_layers=3,
    context_length=128,
  
    use_rope=False
).to(device),
    
    "lm_pfk":GFAPMiniLM_PFK(
    vocab_size=3000,
    d_model=16,
    num_layers=3,
    context_length=128,
  
    use_rope=False
).to(device),

    "lm_base":BaseMiniLM(
    vocab_size=3000,
    d_model=16,
    num_layers=3,
    context_length=128,
    num_heads=8,
    rope_theta=1000).to(device)

    }

### no base

In [4]:
models={
    "lm_po":GFAPMiniLM_PO(
    vocab_size=3000,
    d_model=16,
    num_layers=3,
    context_length=128,
  
    use_rope=False
).to(device),
  "lm_pf": GFAPMiniLM_PF(
    vocab_size=3000,
    d_model=16,
    num_layers=3,
    context_length=128,
  
    use_rope=False
).to(device)
,
    "lm_pok":GFAPMiniLM_POK(
    vocab_size=3000,
    d_model=16,
    num_layers=3,
    context_length=128,
  
    use_rope=False
).to(device),
    
    "lm_pfk":GFAPMiniLM_PFK(
    vocab_size=3000,
    d_model=16,
    num_layers=3,
    context_length=128,
  
    use_rope=False
).to(device)}

## with rope

In [4]:
models={
    "lm_po":GFAPMiniLM_PO(
    vocab_size=3000,
    d_model=16,
    num_layers=3,
    context_length=128,
  
    use_rope=True
).to(device),
  "lm_pf": GFAPMiniLM_PF(
    vocab_size=3000,
    d_model=16,
    num_layers=3,
    context_length=128,
  
    use_rope=True
).to(device)
,
    "lm_pok":GFAPMiniLM_POK(
    vocab_size=3000,
    d_model=16,
    num_layers=3,
    context_length=128,
  
    use_rope=True
).to(device),
    
    "lm_pfk":GFAPMiniLM_PFK(
    vocab_size=3000,
    d_model=16,
    num_layers=3,
    context_length=128,
  
    use_rope=True,
).to(device)

    }

## print mib/params

In [5]:
for name,model in models.items():
    print_model_mib(model,name)

=== lm_po ===
Parameters : 108,724
Param size : 0.415 MiB
Buffer size: 0.000 MiB
Total size : 0.415 MiB
=== lm_pf ===
Parameters : 108,724
Param size : 0.415 MiB
Buffer size: 0.000 MiB
Total size : 0.415 MiB
=== lm_pok ===
Parameters : 108,724
Param size : 0.415 MiB
Buffer size: 0.000 MiB
Total size : 0.415 MiB
=== lm_pfk ===
Parameters : 108,724
Param size : 0.415 MiB
Buffer size: 0.000 MiB
Total size : 0.415 MiB
=== lm_base ===
Parameters : 108,724
Param size : 0.415 MiB
Buffer size: 0.001 MiB
Total size : 0.416 MiB


In [6]:
for name,model in models.items():
    print_param_table(model)

token_embeddings                              48000 params |  0.183 MB
layers.0.ln1                                     16 params |  0.000 MB
layers.0.ln2                                     16 params |  0.000 MB
layers.0.attn.wq                                272 params |  0.001 MB
layers.0.attn.wk                                272 params |  0.001 MB
layers.0.attn.wv                                272 params |  0.001 MB
layers.0.attn.proj                              272 params |  0.001 MB
layers.0.ffn.w1                                 714 params |  0.003 MB
layers.0.ffn.w2                                 688 params |  0.003 MB
layers.0.ffn.w3                                 714 params |  0.003 MB
layers.1.ln1                                     16 params |  0.000 MB
layers.1.ln2                                     16 params |  0.000 MB
layers.1.attn.wq                                272 params |  0.001 MB
layers.1.attn.wk                                272 params |  0.001 MB
layers

# exp

## no-rope

In [7]:
runner = LMExperimentRunner(
    models=models,
    train_dataset=train_ds,
    val_dataset=val_ds,
    num_epochs=20,
    batch_size=32,
    lr=3e-4,
    vocab_size=vocab_size,
)
logs=runner.run()


>>> Model: lm_po
lm_po | Epoch 1 | TrainLoss=6.8970 | ValLoss=6.4712 | PPL=646.24
lm_po | Epoch 2 | TrainLoss=6.4377 | ValLoss=6.3943 | PPL=598.40
lm_po | Epoch 3 | TrainLoss=6.3418 | ValLoss=6.2895 | PPL=538.86
lm_po | Epoch 4 | TrainLoss=6.2413 | ValLoss=6.2046 | PPL=495.04
lm_po | Epoch 5 | TrainLoss=6.1504 | ValLoss=6.1067 | PPL=448.84
lm_po | Epoch 6 | TrainLoss=6.0584 | ValLoss=6.0229 | PPL=412.75
lm_po | Epoch 7 | TrainLoss=5.9862 | ValLoss=5.9661 | PPL=389.99
lm_po | Epoch 8 | TrainLoss=5.9367 | ValLoss=5.9257 | PPL=374.55
lm_po | Epoch 9 | TrainLoss=5.9001 | ValLoss=5.8963 | PPL=363.68
lm_po | Epoch 10 | TrainLoss=5.8721 | ValLoss=5.8725 | PPL=355.13
lm_po | Epoch 11 | TrainLoss=5.8482 | ValLoss=5.8530 | PPL=348.29
lm_po | Epoch 12 | TrainLoss=5.8278 | ValLoss=5.8341 | PPL=341.76
lm_po | Epoch 13 | TrainLoss=5.8097 | ValLoss=5.8217 | PPL=337.54
lm_po | Epoch 14 | TrainLoss=5.7940 | ValLoss=5.8037 | PPL=331.52
lm_po | Epoch 15 | TrainLoss=5.7785 | ValLoss=5.7899 | PPL=326.97
l

In [8]:
save_experiment_csv(logs, "experiment_logs_so_ko_lm.csv")

📁 Saved experiment results to: experiment_logs_so_ko_lm.csv


In [8]:
save_experiment_csv(logs, "experiment_logs_so_ko_lm_gelu.csv")

📁 Saved experiment results to: experiment_logs_so_ko_lm_gelu.csv


In [8]:
save_experiment_csv(logs, "experiment_logs_so_ko_lm_nosqrt.csv")

📁 Saved experiment results to: experiment_logs_so_ko_lm_nosqrt.csv


## with rope


In [5]:
for name,model in models.items():
    print_model_mib(model,name)

=== lm_po ===
Parameters : 108,724
Param size : 0.415 MiB
Buffer size: 0.008 MiB
Total size : 0.423 MiB
=== lm_pf ===
Parameters : 108,724
Param size : 0.415 MiB
Buffer size: 0.008 MiB
Total size : 0.423 MiB
=== lm_pok ===
Parameters : 108,724
Param size : 0.415 MiB
Buffer size: 0.008 MiB
Total size : 0.423 MiB
=== lm_pfk ===
Parameters : 108,724
Param size : 0.415 MiB
Buffer size: 0.008 MiB
Total size : 0.423 MiB


In [6]:
for name,model in models.items():
    print_param_table(model)

token_embeddings                              48000 params |  0.183 MB
layers.0.ln1                                     16 params |  0.000 MB
layers.0.ln2                                     16 params |  0.000 MB
layers.0.attn.wq                                272 params |  0.001 MB
layers.0.attn.wk                                272 params |  0.001 MB
layers.0.attn.wv                                272 params |  0.001 MB
layers.0.attn.proj                              272 params |  0.001 MB
layers.0.ffn.w1                                 714 params |  0.003 MB
layers.0.ffn.w2                                 688 params |  0.003 MB
layers.0.ffn.w3                                 714 params |  0.003 MB
layers.1.ln1                                     16 params |  0.000 MB
layers.1.ln2                                     16 params |  0.000 MB
layers.1.attn.wq                                272 params |  0.001 MB
layers.1.attn.wk                                272 params |  0.001 MB
layers

In [7]:
runner = LMExperimentRunner(
    models=models,
    train_dataset=train_ds,
    val_dataset=val_ds,
    num_epochs=20,
    batch_size=32,
    lr=3e-4,
    vocab_size=vocab_size,
)
logs=runner.run()


>>> Model: lm_po
lm_po | Epoch 1 | TrainLoss=6.9206 | ValLoss=6.4718 | PPL=646.67
lm_po | Epoch 2 | TrainLoss=6.4407 | ValLoss=6.4016 | PPL=602.81
lm_po | Epoch 3 | TrainLoss=6.3463 | ValLoss=6.2913 | PPL=539.84
lm_po | Epoch 4 | TrainLoss=6.2459 | ValLoss=6.2122 | PPL=498.78
lm_po | Epoch 5 | TrainLoss=6.1772 | ValLoss=6.1488 | PPL=468.18
lm_po | Epoch 6 | TrainLoss=6.1025 | ValLoss=6.0693 | PPL=432.38
lm_po | Epoch 7 | TrainLoss=6.0262 | ValLoss=6.0061 | PPL=405.90
lm_po | Epoch 8 | TrainLoss=5.9715 | ValLoss=5.9613 | PPL=388.13
lm_po | Epoch 9 | TrainLoss=5.9307 | ValLoss=5.9260 | PPL=374.67
lm_po | Epoch 10 | TrainLoss=5.8969 | ValLoss=5.8991 | PPL=364.71
lm_po | Epoch 11 | TrainLoss=5.8682 | ValLoss=5.8731 | PPL=355.36
lm_po | Epoch 12 | TrainLoss=5.8425 | ValLoss=5.8520 | PPL=347.92
lm_po | Epoch 13 | TrainLoss=5.8191 | ValLoss=5.8304 | PPL=340.48
lm_po | Epoch 14 | TrainLoss=5.7982 | ValLoss=5.8137 | PPL=334.86
lm_po | Epoch 15 | TrainLoss=5.7794 | ValLoss=5.7974 | PPL=329.46
l

In [8]:
save_experiment_csv(logs, "experiment_logs_so_ko_lm_rope.csv")

📁 Saved experiment results to: experiment_logs_so_ko_lm_rope.csv
